<a href="https://colab.research.google.com/github/ksaini762/BhashaBot/blob/main/BhashaBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install all required packages
!pip install -q langchain langchain-community langchain-groq
!pip install -q chromadb sentence-transformers
!pip install -q pypdf unstructured
!pip install -q deep-translator
print('All packages installed successfully!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [3]:
import os

GROQ_API_KEY = "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"

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print('API Key set!')

API Key set!


In [4]:
from google.colab import files
import os

os.makedirs('/content/documents', exist_ok=True)

print('Please upload your PDF or TXT documents:')
uploaded = files.upload()

for filename in uploaded.keys():
    dest = f'/content/documents/{filename}'
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f'Saved: {filename}')

print(f'Total files uploaded: {len(uploaded)}')

Please upload your PDF or TXT documents:


Saving harrypotter.pdf to harrypotter.pdf
Saved: harrypotter.pdf
Total files uploaded: 1


In [7]:
!pip install -q langchain-huggingface

In [8]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

print('Loading documents...')

all_documents = []

for filename in os.listdir('/content/documents'):
    filepath = f'/content/documents/{filename}'
    try:
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(filepath)
        elif filename.endswith('.txt'):
            loader = TextLoader(filepath)
        else:
            print(f'Skipping: {filename}')
            continue
        docs = loader.load()
        all_documents.extend(docs)
        print(f'Loaded: {filename} ({len(docs)} pages)')
    except Exception as e:
        print(f'Error loading {filename}: {e}')

print(f'Total pages loaded: {len(all_documents)}')

print('Splitting into chunks...')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(all_documents)
print(f'Total chunks: {len(chunks)}')

print('Creating embeddings (takes 1-2 mins)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print('Storing in vector database...')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='/content/chroma_db'
)

print('Knowledge base ready!')

Loading documents...
Loaded: harrypotter.pdf (3623 pages)
Total pages loaded: 3623
Splitting into chunks...
Total chunks: 15347
Creating embeddings (takes 1-2 mins)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Storing in vector database...
Knowledge base ready!


In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from deep_translator import GoogleTranslator

llm = ChatGroq(
    model_name='llama-3.3-70b-versatile',
    temperature=0.3,
    api_key=GROQ_API_KEY
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

prompt_template = PromptTemplate.from_template("""
You are a helpful assistant. Use the context below to answer the question.
Give a clear and short answer. If the answer is not in the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

def translate_to_punjabi(text):
    try:
        return GoogleTranslator(source='auto', target='pa').translate(text)
    except Exception as e:
        return f'[Translation error: {e}]'

def boliai_chat(user_question):
    print(f'You asked: {user_question}')
    print('Searching documents...')
    english_answer = chain.invoke(user_question)
    print(f'\nEnglish Answer:\n{english_answer}')
    print('\nTranslating to Punjabi...')
    punjabi_answer = translate_to_punjabi(english_answer)
    print(f'\nPunjabi Answer:\n{punjabi_answer}')
    print('\n' + '='*60)
    return english_answer, punjabi_answer

print('Chatbot is ready!')

Chatbot is ready!


In [18]:
# Question 1
boliai_chat("What is this project about?")

You asked: What is this project about?
Searching documents...

English Answer:
I don't know based on the provided documents.

Translating to Punjabi...

Punjabi Answer:
ਮੈਨੂੰ ਮੁਹੱਈਆ ਕੀਤੇ ਦਸਤਾਵੇਜ਼ਾਂ ਦੇ ਆਧਾਰ 'ਤੇ ਪਤਾ ਨਹੀਂ ਹੈ।



("I don't know based on the provided documents.",
 "ਮੈਨੂੰ ਮੁਹੱਈਆ ਕੀਤੇ ਦਸਤਾਵੇਜ਼ਾਂ ਦੇ ਆਧਾਰ 'ਤੇ ਪਤਾ ਨਹੀਂ ਹੈ।")

In [19]:
# Question 2
boliai_chat("What are the main technologies used?")

You asked: What are the main technologies used?
Searching documents...

English Answer:
I don't know based on the provided documents.

Translating to Punjabi...

Punjabi Answer:
ਮੈਨੂੰ ਮੁਹੱਈਆ ਕੀਤੇ ਦਸਤਾਵੇਜ਼ਾਂ ਦੇ ਆਧਾਰ 'ਤੇ ਪਤਾ ਨਹੀਂ ਹੈ।



("I don't know based on the provided documents.",
 "ਮੈਨੂੰ ਮੁਹੱਈਆ ਕੀਤੇ ਦਸਤਾਵੇਜ਼ਾਂ ਦੇ ਆਧਾਰ 'ਤੇ ਪਤਾ ਨਹੀਂ ਹੈ।")

In [20]:
# Question 3
boliai_chat("What are the outcomes of this project?")

You asked: What are the outcomes of this project?
Searching documents...

English Answer:
I don't know based on the provided documents.

Translating to Punjabi...

Punjabi Answer:
ਮੈਨੂੰ ਮੁਹੱਈਆ ਕੀਤੇ ਦਸਤਾਵੇਜ਼ਾਂ ਦੇ ਆਧਾਰ 'ਤੇ ਪਤਾ ਨਹੀਂ ਹੈ।



("I don't know based on the provided documents.",
 "ਮੈਨੂੰ ਮੁਹੱਈਆ ਕੀਤੇ ਦਸਤਾਵੇਜ਼ਾਂ ਦੇ ਆਧਾਰ 'ਤੇ ਪਤਾ ਨਹੀਂ ਹੈ।")

In [ ]:
print('BOLIAI is ready! Type your question and press Enter.')
print('Type exit to stop.')
print('='*60)

while True:
    user_input = input('\nYour Question: ')
    if user_input.lower() in ['exit', 'quit', 'bye']:
        print('Goodbye! Thank you!')
        break
    if user_input.strip() == '':
        print('Please enter a question.')
        continue
    boliai_chat(user_input)

BOLIAI is ready! Type your question and press Enter.
Type exit to stop.

Your Question: Who is Hermione Granger?
You asked: Who is Hermione Granger?
Searching documents...

English Answer:
Hermione Granger is a friend of the narrator and others, who became friends with them after they worked together to knock out a twelve-foot mountain troll.

Translating to Punjabi...

Punjabi Answer:
ਹਰਮਾਇਓਨ ਗ੍ਰੇਂਜਰ ਕਥਾਵਾਚਕ ਅਤੇ ਹੋਰਾਂ ਦਾ ਇੱਕ ਦੋਸਤ ਹੈ, ਜੋ ਬਾਰਾਂ ਫੁੱਟ ਪਹਾੜੀ ਟ੍ਰੋਲ ਨੂੰ ਖੜਕਾਉਣ ਲਈ ਇਕੱਠੇ ਕੰਮ ਕਰਨ ਤੋਂ ਬਾਅਦ ਉਹਨਾਂ ਦੇ ਦੋਸਤ ਬਣ ਗਏ।


Your Question: What is Hogwarts?
You asked: What is Hogwarts?
Searching documents...

English Answer:
Hogwarts is a historic school.

Translating to Punjabi...

Punjabi Answer:
ਹੌਗਵਾਰਟਸ ਇੱਕ ਇਤਿਹਾਸਕ ਸਕੂਲ ਹੈ।


Your Question: Who is Voldemort?
You asked: Who is Voldemort?
Searching documents...

English Answer:
Lord Voldemort is not clearly defined in the context, but it is mentioned that he is someone who was trying to get a job at Hogwarts, possibly as a teacher, and is 